# AI CUP 2026 桌球預測 — main.ipynb (v14 FINAL)

> **Private LB 第 5 名 (0.3890227 / 423 隊)** 的最終版本
> Public LB 第 45 名 (0.4341008)
> 14 次提交產出此結果

## 整體流程

```
train.csv + test_new.csv + 舊測試集 (Reference Only)
    ↓
[1] 特徵工程 + 樣本構造(prefix → next stroke)
    ↓
[2] Fold-safe 統計量(用 train+old 擴充)
    - player_dists: P(next_action | next_hitter)
    - matchup: P(next_action | next_hitter, opp_cluster)
    - transition: P(next_action | last_a, last_p)
    ↓
[3] 噪音過濾(剔除 1053 個 prob_true<5% 的 noisy rallies)
    ↓
[4] ★ Priority 3:把 old test 加進訓練集(不只當統計 prior)
    ↓
[5] 3-model ensemble:
    - LGBM (action with aug features, point/server clean)
    - TabPFN (same task gating)
    - GRU = 0.5 × (v7 簡單 GRU × 3 seeds) + 0.5 × (v8 Time2Vec+FiLM GRU × 3 seeds)
    ↓
[6] Server override(用舊測試集的真值覆蓋 1236 個公開 rally — README 明文允許)
    ↓
submission_FINAL.csv → 上傳
```

## 關鍵設計決策(同學請特別注意)

1. **Task-specific feature gating**:不是所有 task 都吃同樣 features
   - action 用 train+old augmented features(player style 跨場次可轉移)
   - point/server 用 train-only features(point 太依賴對手/局勢,augmented 反而傷)

2. **Multi-architecture diversity**:v7 + v8 兩個 GRU 結構,blend at α=0.5
   - v7:簡單 GRU(linear num projection)
   - v8:Time2Vec 時序嵌入 + FiLM 條件化(對 train 學的不一樣)
   - Blend 後 rescued action F1 大幅提升 +0.043

3. **Multi-seed averaging**:每架構 3 seeds 平均,降低 variance

4. **Noise filter**:用 v10 ensemble OOF 找 prob_true_action<5% 的 rally
   (這些是「不可能對的」訓練樣本,剔除後 model 不會浪費容量擬合垃圾)

5. **Priority 3**:舊測試集 1236 rally 整個加進訓練集(README 允許)
   - LGBM all-prefix +2353 樣本
   - TabPFN sampled +838 樣本
   - GRU all-prefix +2353 樣本


In [ ]:
# === 設定 ===
import time, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import GroupKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score
import lightgbm as lgb, torch, torch.nn as nn
from tabpfn import TabPFNClassifier
from tabpfn_extensions.many_class import ManyClassClassifier

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)

## 1. 資料載入

- `train.csv` — 主要訓練資料(216 場比賽)
- `test_new.csv` — 評測用測試集(79 場;55 場與 old 重疊 + 24 場全新)
- `Reference_Only_Old_Test_Data/test.csv` — 舊版測試集(README 明文允許使用)
  - 含 serverGetPoint 欄位(被 leak 出去的答案)
  - 但 actionId/pointId 是「prefix 觀察」非答案,跟新測試集完全一樣


In [ ]:
tr  = pd.read_csv('../data/train.csv')
te  = pd.read_csv('../data/test_new.csv')
old = pd.read_csv('../data/Reference_Only_Old_Test_Data/test.csv')
print(f"train: {len(tr):>6} rows | {tr.rally_uid.nunique():>5} rallies | {tr.match.nunique():>3} matches")
print(f"test : {len(te):>6} rows | {te.rally_uid.nunique():>5} rallies | {te.match.nunique():>3} matches")
print(f"old  : {len(old):>6} rows | {old.rally_uid.nunique():>5} rallies | {old.match.nunique():>3} matches")
print(f"\nold∩new rally_uid: {len(set(old.rally_uid)&set(te.rally_uid))}  (= public leaked subset)")
print(f"new-only rallies : {te.rally_uid.nunique()-len(set(old.rally_uid)&set(te.rally_uid))}  (= private clean subset = 24 new matches)")

## 2. 樣本構造

把每個 rally(按 strikeNumber 排序)切成「觀察 prefix L → 預測第 L+1 拍」。
- 訓練用 all-prefix(L=1..T-1)
- OOF eval 用 test 長度分佈隨機抽 1 個 L(模擬排行榜情境)

關鍵設計:**build() 函式有 `exclude_uids` 參數**支援 v12 的噪音過濾。


In [ ]:
STROKE = ["strikeId","handId","strengthId","spinId","pointId","actionId","positionId"]
REC    = ["scoreSelf","scoreOther"] + STROKE
ACLS   = np.arange(19); PCLS = np.arange(10)

def feats(strokes, sex, L):
    last = strokes[L-1]
    f = {"sex":sex, "obs_len":L, "obs_parity":L%2, "next_is_server":(L+1)%2,
         "score_self":last["scoreSelf"], "score_other":last["scoreOther"],
         "score_diff":last["scoreSelf"]-last["scoreOther"], "score_sum":last["scoreSelf"]+last["scoreOther"]}
    for off in range(1, 4):
        tag = "last" if off == 1 else f"lag{off}"
        if L >= off:
            r = strokes[L-off]
            for c in STROKE: f[f"{tag}_{c}"] = r[c]
        else:
            for c in STROKE: f[f"{tag}_{c}"] = -1
    f["mean_spin"]     = float(np.mean([strokes[i]["spinId"]     for i in range(L)]))
    f["mean_strength"] = float(np.mean([strokes[i]["strengthId"] for i in range(L)]))
    f["nuniq_point"]   = len({strokes[i]["pointId"]  for i in range(L)})
    f["nuniq_action"]  = len({strokes[i]["actionId"] for i in range(L)})
    f["_la"] = last["actionId"]; f["_lp"] = last["pointId"]
    return f

def build(df, mode, tld, seed=SEED, test_mode=False, exclude_uids=None):
    """建構 prefix→next 樣本。exclude_uids 用來排除噪音 rally(v12 feature)"""
    rng = np.random.default_rng(seed)
    if mode == "sampled":
        Ls = np.array(sorted(tld)); Ps = np.array([tld[l] for l in Ls], float); Ps /= Ps.sum()
    rows, yA, yP, yR, nh, lh, uid, g = [], [], [], [], [], [], [], []
    for rid, grp in df.groupby("rally_uid", sort=False):
        if (exclude_uids is not None) and (int(rid) in exclude_uids): continue   # ★ 噪音過濾
        grp = grp.sort_values("strikeNumber"); T = len(grp)
        st = grp[REC].to_dict("records")
        gp = grp.gamePlayerId.to_numpy(); go = grp.gamePlayerOtherId.to_numpy()
        if test_mode:
            Ll = [T]
        else:
            if T < 2: continue
            na  = grp.actionId.to_numpy(); npt = grp.pointId.to_numpy()
            sgp = int(grp.serverGetPoint.iloc[0]); mt = int(grp.match.iloc[0])
            Ll = range(1, T) if mode == "all" else (
                [1] if len(Ls[Ls<=T-1])==0
                else [int(rng.choice(Ls[Ls<=T-1], p=(Ps[Ls<=T-1]/Ps[Ls<=T-1].sum())))]
            )
        for L in Ll:
            rows.append(feats(st, int(grp.sex.iloc[0]), L))
            nh.append(int(go[L-1])); lh.append(int(gp[L-1]))
            if test_mode: uid.append(int(rid))
            else: yA.append(int(na[L])); yP.append(int(npt[L])); yR.append(sgp); g.append(mt)
    X = pd.DataFrame(rows)
    if test_mode: return X, np.array(nh), np.array(lh), np.array(uid)
    return X, np.array(yA), np.array(yP), np.array(yR), np.array(nh), np.array(lh), np.array(g)

## 3. Fold-safe 統計量

4 個統計量,**每個 fold 都用「fold-train + 全部 old」重新算**(old 跟 train 完全沒重疊,fold-safe):

- **`fit_trans`**:`P(next_action | last_a, last_p)` 動作鏈
- **`player_dists`**:`P(next_action | next_hitter)` ← 冷啟動關鍵
- **`fit_clusters`**:KMeans k=6 把選手分成 6 種風格
- **`fit_matchup`**:`P(next_action | next_hitter, opp_cluster)` 配對風格

★ **這套統計量用 train+old 擴充過**(讓 test 的「沒見過選手」也有 prior)


In [ ]:
def fit_trans(keys, yA, yP, alpha=1.0):
    def cd_(ka, y, nc):
        d = {}
        for k, yy in zip(ka, y): d.setdefault(k, np.zeros(nc))[yy] += 1
        gp = np.bincount(y, minlength=nc) + alpha; gp /= gp.sum()
        return {k:(v+alpha)/(v.sum()+alpha*nc) for k, v in d.items()}, gp
    lalp = list(zip(keys["_la"], keys["_lp"]))
    return dict(aJ=cd_(lalp, yA, 19), pJ=cd_(lalp, yP, 10))

def apply_trans(keys, T):
    out = {}; lalp = list(zip(keys["_la"], keys["_lp"]))
    d, gp = T["aJ"]; M = np.array([d.get(k, gp) for k in lalp])
    for j in range(19): out[f"tA_{j}"] = M[:, j]
    d, gp = T["pJ"]; M = np.array([d.get(k, gp) for k in lalp])
    for j in range(10): out[f"tP_{j}"] = M[:, j]
    return pd.DataFrame(out)

def player_dists(nh, yA, yP, alpha=10):
    dA, dP = {}, {}
    for h, a in zip(nh, yA): dA.setdefault(h, np.zeros(19))[a] += 1
    for h, p in zip(nh, yP): dP.setdefault(h, np.zeros(10))[p] += 1
    gA = np.bincount(yA, minlength=19) + 1.; gA /= gA.sum()
    gP = np.bincount(yP, minlength=10) + 1.; gP /= gP.sum()
    return ({h:(v+alpha*gA)/(v.sum()+alpha) for h, v in dA.items()}, gA,
            {h:(v+alpha*gP)/(v.sum()+alpha) for h, v in dP.items()}, gP)

def fit_clusters(df_sub, k=6):
    vecs = {}
    for pl, g in df_sub.groupby('gamePlayerId'):
        a = np.bincount(g.actionId, minlength=19).astype(float); a /= max(a.sum(), 1)
        p = np.bincount(g.pointId,  minlength=10).astype(float); p /= max(p.sum(), 1)
        vecs[pl] = np.concatenate([a, p])
    pls = list(vecs)
    km  = KMeans(k, n_init=10, random_state=SEED).fit(np.array([vecs[p] for p in pls]))
    return {p:int(c) for p, c in zip(pls, km.labels_)}

def fit_matchup(nh, lh, yA, yP, cl):
    cA, cP = {}, {}
    for h, o, a, p in zip(nh, [cl.get(x, -1) for x in lh], yA, yP):
        cA.setdefault((h, o), np.zeros(19))[a] += 1
        cP.setdefault((h, o), np.zeros(10))[p] += 1
    return cA, cP

def matchup_feat(nh, lh, cl, cA, cP, dMa, gA, dMp, gP, idx, a=8):
    ocs = [cl.get(x, -1) for x in lh]
    MA = np.array([(cA.get((h,o), np.zeros(19)) + a*dMa.get(h, gA)) / (cA.get((h,o), np.zeros(19)).sum() + a) for h, o in zip(nh, ocs)])
    MP = np.array([(cP.get((h,o), np.zeros(10)) + a*dMp.get(h, gP)) / (cP.get((h,o), np.zeros(10)).sum() + a) for h, o in zip(nh, ocs)])
    return pd.DataFrame({**{f'mcA{j}':MA[:,j] for j in range(19)}, **{f'mcP{j}':MP[:,j] for j in range(10)}}, index=idx)

def player_feat(nh, dA, gA, dP, gP, idx):
    MA = np.array([dA.get(h, gA) for h in nh])
    MP = np.array([dP.get(h, gP) for h in nh])
    return pd.DataFrame({**{f'phA{j}':MA[:,j] for j in range(19)}, **{f'phP{j}':MP[:,j] for j in range(10)}}, index=idx)

## 4. 超參數(集中管理)

In [ ]:
# === LGBM ===
LGBM_PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=63,
                   subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
def lgbc(balanced=True):
    p = dict(LGBM_PARAMS)
    if balanced: p["class_weight"] = "balanced"
    return lgb.LGBMClassifier(**p)

# === GRU ===
GRU_HIDDEN  = 64
GRU_DROPOUT = 0.2
GRU_LR      = 1e-3
GRU_EPOCHS  = 12
GRU_BATCH   = 256
GRU_LOSS_W  = (0.4, 0.4, 0.2)   # 對齊最終分數權重
SEEDS       = [42, 1, 2]         # multi-seed averaging

# === TabPFN ===
TABPFN_MANY_CLASS_ALPHA = 10

# === Ensemble / 決策 ===
BETA_GRID   = np.linspace(0, 2.5, 21)
WEIGHT_STEP = 0.05

# === ★ Task-specific feature gating ===
# Per-stratum 證據:
#   - rescued action F1: aug 0.352 vs train-only ~0.28(差 0.07)→ action 用 aug
#   - rescued point  F1: aug 0.120 vs train-only 0.168(aug 反而傷)→ point 用 clean
ACTION_FEATURE_SOURCE = "train+old"
POINT_FEATURE_SOURCE  = "train-only"
SERVER_FEATURE_SOURCE = "train-only"

# === v12 噪音過濾門檻 ===
NOISE_THRESHOLD = 0.05   # 移除 v10 ensemble 預測 prob_true_action < 5% 的 rally
GRU_BLEND       = 0.5    # v7 GRU 與 v8 GRU 的 blend ratio
print("hyperparams loaded")

## 5. GRU 模型(v7 baseline + v8 Time2Vec+FiLM)

兩個架構**故意設計成不同**,blend 後互補:

- **v7 GRU**:簡單 linear projection 處理數值特徵
- **v8 GRU**:Time2Vec(周期時序嵌入)+ FiLM(條件化最終 hidden state)

兩個架構單獨表現差不多,但 blend 後 rescued action F1 從 0.352 跳到 0.395(+0.043)


In [ ]:
CAT = ["actionId","pointId","spinId","strengthId","handId","positionId","strikeId"]
VOCAB = {c: int(tr[c].max())+2 for c in CAT}; VOCAB['role'] = 3; VOCAB['sex'] = int(tr.sex.max()) + 2
NCAT = len(CAT) + 2; MAXLEN = 30

def rseq(g):
    g = g.sort_values("strikeNumber")
    cat = np.stack([g[c].to_numpy()+1 for c in CAT] +
                   [(g.strikeNumber.to_numpy()%2)+1,
                    np.full(len(g), int(g.sex.iloc[0])+1)], axis=1)
    num = np.stack([g.scoreSelf.to_numpy()/10.,
                    g.scoreOther.to_numpy()/10.,
                    g.strikeNumber.to_numpy()/15.], axis=1)
    sgp = int(g.serverGetPoint.iloc[0]) if "serverGetPoint" in g.columns else 0
    return cat.astype(np.int64), num.astype(np.float32), g.actionId.to_numpy(), g.pointId.to_numpy(), sgp

def build_seq(df, mode, tld, seed=SEED, test_mode=False, exclude_uids=None):
    rng = np.random.default_rng(seed)
    if mode == "sampled":
        Ls = np.array(sorted(tld)); Ps = np.array([tld[l] for l in Ls], float); Ps /= Ps.sum()
    C, Nu, Ln, yA, yP, yR = [], [], [], [], [], []
    for rid, grp in df.groupby("rally_uid", sort=False):
        if (exclude_uids is not None) and (int(rid) in exclude_uids): continue
        cat, num, na, npt, sgp = rseq(grp); T = len(na)
        if test_mode: Ll = [T]
        else:
            if T < 2: continue
            Ll = range(1, T) if mode == "all" else (
                [1] if len(Ls[Ls<=T-1])==0
                else [int(rng.choice(Ls[Ls<=T-1], p=(Ps[Ls<=T-1]/Ps[Ls<=T-1].sum())))]
            )
        for L in Ll:
            l = min(L, MAXLEN); pc = np.zeros((MAXLEN, NCAT), np.int64); pn = np.zeros((MAXLEN, 3), np.float32)
            pc[:l] = cat[L-l:L]; pn[:l] = num[L-l:L]
            C.append(pc); Nu.append(pn); Ln.append(l)
            if not test_mode: yA.append(int(na[L])); yP.append(int(npt[L])); yR.append(sgp)
    if test_mode: return np.stack(C), np.stack(Nu), np.array(Ln)
    return np.stack(C), np.stack(Nu), np.array(Ln), np.array(yA), np.array(yP), np.array(yR)

class GRUNetV7(nn.Module):
    """v7 baseline GRU(linear projection for numerics)"""
    def __init__(s):
        super().__init__()
        s.embs = nn.ModuleList(
            [nn.Embedding(VOCAB[c], 8, padding_idx=0) for c in CAT] +
            [nn.Embedding(VOCAB['role'], 4, padding_idx=0),
             nn.Embedding(VOCAB['sex'],  4, padding_idx=0)])
        s.num  = nn.Linear(3, 16)
        s.gru  = nn.GRU(8*len(CAT) + 4 + 4 + 16, GRU_HIDDEN, batch_first=True)
        s.drop = nn.Dropout(GRU_DROPOUT)
        s.ha   = nn.Linear(GRU_HIDDEN, 19)
        s.hp   = nn.Linear(GRU_HIDDEN, 10)
        s.hs   = nn.Linear(GRU_HIDDEN, 1)
    def forward(s, cat, num, ln):
        e = torch.cat([s.embs[i](cat[:,:,i]) for i in range(NCAT)] + [torch.relu(s.num(num))], -1)
        pk = nn.utils.rnn.pack_padded_sequence(e, ln.cpu(), batch_first=True, enforce_sorted=False)
        _, h = s.gru(pk); h = s.drop(h[-1])
        return s.ha(h), s.hp(h), s.hs(h).squeeze(1)

class Time2Vec(nn.Module):
    """Time2Vec layer: 每個 scalar → (k+1)-dim(1 linear + k periodic sin)"""
    def __init__(s, in_dim, k=7):
        super().__init__()
        s.w_lin = nn.Parameter(torch.randn(in_dim) * 0.1)
        s.b_lin = nn.Parameter(torch.zeros(in_dim))
        s.W     = nn.Parameter(torch.randn(in_dim, k) * 0.5)
        s.B     = nn.Parameter(torch.randn(in_dim, k) * 0.5)
    def forward(s, x):
        lin = (x * s.w_lin + s.b_lin).unsqueeze(-1)
        per = torch.sin(x.unsqueeze(-1) * s.W + s.B)
        return torch.cat([lin, per], -1).flatten(-2)

class FiLM(nn.Module):
    """FiLM: gamma, beta = MLP(context); feat ← feat * (1+tanh(gamma)) + beta"""
    def __init__(s, ctx_dim, feat_dim):
        super().__init__(); s.proj = nn.Linear(ctx_dim, feat_dim*2)
    def forward(s, feat, ctx):
        gb = s.proj(ctx); g, b = gb.chunk(2, -1)
        return feat * (1.0 + torch.tanh(g)) + b

class GRUNetV8(nn.Module):
    """v8 GRU = Time2Vec + FiLM 條件化(故意跟 v7 學不一樣的東西)"""
    def __init__(s):
        super().__init__()
        s.embs = nn.ModuleList(
            [nn.Embedding(VOCAB[c], 8, padding_idx=0) for c in CAT] +
            [nn.Embedding(VOCAB['role'], 4, padding_idx=0),
             nn.Embedding(VOCAB['sex'],  4, padding_idx=0)])
        s.t2v  = Time2Vec(3, k=7)                # 3 個 scalar × (1+7) = 24 dim
        s.gru  = nn.GRU(8*len(CAT) + 4 + 4 + 24, GRU_HIDDEN, batch_first=True)
        s.drop = nn.Dropout(GRU_DROPOUT)
        s.film = FiLM(ctx_dim=4+24+4, feat_dim=GRU_HIDDEN)
        s.ha   = nn.Linear(GRU_HIDDEN, 19)
        s.hp   = nn.Linear(GRU_HIDDEN, 10)
        s.hs   = nn.Linear(GRU_HIDDEN, 1)
    def forward(s, cat, num, ln):
        emb_list = [s.embs[i](cat[:,:,i]) for i in range(NCAT)]
        t2v = s.t2v(num)
        e = torch.cat(emb_list + [t2v], -1)
        pk = nn.utils.rnn.pack_padded_sequence(e, ln.cpu(), batch_first=True, enforce_sorted=False)
        _, h = s.gru(pk); h = s.drop(h[-1])
        sex_emb  = s.embs[NCAT-1](cat[:, 0, NCAT-1])
        role_emb = s.embs[NCAT-2](cat[:, :, NCAT-2]).mean(1)
        mask = (torch.arange(t2v.size(1), device=t2v.device).unsqueeze(0) < ln.unsqueeze(1)).unsqueeze(-1).float()
        t2v_pool = (t2v*mask).sum(1) / mask.sum(1).clamp_min(1.0)
        ctx = torch.cat([sex_emb, t2v_pool, role_emb], -1)
        h = s.film(h, ctx)
        return s.ha(h), s.hp(h), s.hs(h).squeeze(1)

def _cw(y, n):
    c = np.bincount(y, minlength=n) + 1
    w = 1. / c
    return torch.tensor(w*n/w.sum(), dtype=torch.float32, device=DEV)

def gru_pred(m, Xc, Xn, Xl):
    m.eval(); A, P, R = [], [], []
    with torch.no_grad():
        for i in range(0, len(Xl), 512):
            la, lp, lr = m(torch.tensor(Xc[i:i+512], device=DEV),
                           torch.tensor(Xn[i:i+512], device=DEV),
                           torch.tensor(Xl[i:i+512], device=DEV))
            A.append(torch.softmax(la, 1).cpu().numpy())
            P.append(torch.softmax(lp, 1).cpu().numpy())
            R.append(torch.sigmoid(lr).cpu().numpy())
    return np.vstack(A), np.vstack(P), np.concatenate(R)

def multiseed_train_predict(Cls, Ca, Na, La_, gyA, gyP, gyR, Cte, Nte, Lte, seeds):
    """Multi-seed averaging:訓練 N 個 seed 的同一架構,平均預測"""
    gA_acc, gP_acc, gR_acc = None, None, None
    for s in seeds:
        torch.manual_seed(s); np.random.seed(s)
        m = Cls().to(DEV); opt = torch.optim.Adam(m.parameters(), GRU_LR)
        cea = nn.CrossEntropyLoss(weight=_cw(gyA, 19))
        cep = nn.CrossEntropyLoss(weight=_cw(gyP, 10))
        bce = nn.BCEWithLogitsLoss()
        Cc = torch.tensor(Ca, device=DEV); Nn = torch.tensor(Na, device=DEV); Ll = torch.tensor(La_, device=DEV)
        Ta = torch.tensor(gyA, device=DEV); Tp = torch.tensor(gyP, device=DEV); Tr = torch.tensor(gyR.astype('float32'), device=DEV)
        ii = np.arange(len(gyA))
        for e in range(GRU_EPOCHS):
            m.train(); np.random.shuffle(ii)
            for i in range(0, len(ii), GRU_BATCH):
                b = ii[i:i+GRU_BATCH]; opt.zero_grad()
                la, lp, lr = m(Cc[b], Nn[b], Ll[b])
                wa, wp, ws = GRU_LOSS_W
                (wa*cea(la, Ta[b]) + wp*cep(lp, Tp[b]) + ws*bce(lr, Tr[b])).backward()
                opt.step()
        a, p, r = gru_pred(m, Cte, Nte, Lte)
        gA_acc = a if gA_acc is None else gA_acc + a
        gP_acc = p if gP_acc is None else gP_acc + p
        gR_acc = r if gR_acc is None else gR_acc + r
    n = len(seeds); return gA_acc/n, gP_acc/n, gR_acc/n

## 6. 噪音過濾(v12 feature)

用快取的 v7+v8 OOF ensemble 識別「prob_true_action < 5%」的 rally。
這些是 model 預測「真實 label 機率 < 5%」的 rally,屬於:
1. Label noise(可能是標注錯誤)
2. 極端 outlier
3. 模糊案例

剔除這些 rally,讓 model 訓練時不要浪費容量擬合垃圾。

📊 **效果**:1053/14995 = 7% rally 被剔除,**其中 rescued 子集 noise 率最低**
(4% vs seen 6.4% vs cold 9.8%) → 對私人 rescued 影響極小,**filter 對 private 安全**


In [ ]:
# 載入 v7/v8 OOF 快取(從早期 fold OOF 計算的)
# 同學的話:如果你想自己重跑,需要先跑出 oof_v7_actionaug.npz 和 oof_v8_t2v_film.npz
v7c = np.load('oof_v7_actionaug.npz')
v8c = np.load('oof_v8_t2v_film.npz')

# v10-style ensemble probability(LGBM 0.35 + TabPFN 0.25 + GRU 0.40)
GA_blend = 0.5*v7c['GAc'] + 0.5*v8c['GAc']
PA_oof = 0.35*v7c['LAc'] + 0.25*v7c['TAc'] + 0.40*GA_blend
prob_true = PA_oof[np.arange(len(v7c['YA'])), v7c['YA']]

# 把 OOF index 對應回 rally_uid(透過 fold 順序重建)
tld = te.groupby('rally_uid').size().value_counts().to_dict()
_rng = np.random.default_rng(SEED)
_uid_sampled = []
_Ls = np.array(sorted(tld)); _Ps = np.array([tld[l] for l in _Ls], float); _Ps /= _Ps.sum()
for rid, grp in tr.groupby('rally_uid', sort=False):
    grp = grp.sort_values("strikeNumber"); T = len(grp)
    if T < 2: continue
    cand = _Ls[_Ls <= T-1]
    L = 1 if len(cand)==0 else int(_rng.choice(cand, p=_Ps[_Ls<=T-1]/_Ps[_Ls<=T-1].sum()))
    _uid_sampled.append(int(rid))
_uid_sampled = np.array(_uid_sampled)

# fold 對應 → 重建 OOF concat 順序
_, _, _, _, _, _, _ga = build(tr, "all",     tld)
_, _, _, _, _, _, _gs = build(tr, "sampled", tld)
_M = np.array(sorted(set(_ga) | set(_gs)))
_gkf = GroupKFold(5); _fo = {}
for _f, (_, _vi) in enumerate(_gkf.split(_M, groups=_M)):
    for _m in _M[_vi]: _fo[_m] = _f
_sf = np.array([_fo[m] for m in _gs])
_oof_order = np.concatenate([np.where(_sf == f)[0] for f in range(5)])
_uid_oof = _uid_sampled[_oof_order]

# ★ 識別噪音 rally(prob_true < NOISE_THRESHOLD)
EXCLUDE_UIDS = set(_uid_oof[prob_true < NOISE_THRESHOLD].tolist())
print(f"v12 噪音過濾:{len(EXCLUDE_UIDS)}/{len(_uid_oof)} rally 被排除 (prob_true_action < {NOISE_THRESHOLD})")

## 7. 建構樣本(訓練 + 舊測試 + 預測)

In [ ]:
Xa,  yA,  yP,  yR,  nha,  lha,  ga  = build(tr,  "all",     tld, exclude_uids=EXCLUDE_UIDS)
Xs,  eA,  eP,  eR,  nhs,  lhs,  gs  = build(tr,  "sampled", tld, exclude_uids=EXCLUDE_UIDS)
Xao, yAo, yPo, yRo, nhao, lhao, gao = build(old, "all",     tld)   # ★ OLD 不過濾,整個用
Xao_smp, eAo, ePo, eRo, nhao_s, lhao_s, gao_s = build(old, "sampled", tld)
Xte, nht, lht, uids = build(te, "sampled", tld, test_mode=True)

print(f"train all-prefix:   {len(Xa):>6} samples (filtered)")
print(f"train sampled:      {len(Xs):>6} samples (filtered)")
print(f"old all-prefix:     {len(Xao):>6} samples  ← Priority 3 將加進訓練")
print(f"old sampled:        {len(Xao_smp):>6} samples")
print(f"test:               {len(Xte):>6} samples")

KEY  = ['_la','_lp']
BASE = [c for c in Xa.columns if c not in KEY]

## 8. 全資料 fold-safe 統計(action 用 aug / point+server 用 train-only)

In [ ]:
# === AUG 統計(action 用)= train + old ===
F_la = np.concatenate([Xa["_la"].to_numpy(), Xao["_la"].to_numpy()])
F_lp = np.concatenate([Xa["_lp"].to_numpy(), Xao["_lp"].to_numpy()])
F_yA = np.concatenate([yA, yAo]); F_yP = np.concatenate([yP, yPo])
F_nh = np.concatenate([nha, nhao]); F_lh = np.concatenate([lha, lhao])
T_AUG = fit_trans({"_la":F_la, "_lp":F_lp}, F_yA, F_yP)
dMa_A, gA_A, dMp_A, gP_A = player_dists(F_nh, F_yA, F_yP)
clA = fit_clusters(pd.concat([tr, old], ignore_index=True))
cAA, cPA = fit_matchup(F_nh, F_lh, F_yA, F_yP, clA)

# === TRAIN-ONLY 統計(point + server 用)= 只用 train ===
T_TO = fit_trans({"_la":Xa["_la"].to_numpy(), "_lp":Xa["_lp"].to_numpy()}, yA, yP)
dMa_T, gA_T, dMp_T, gP_T = player_dists(nha, yA, yP)
clT = fit_clusters(tr)
cAT, cPT = fit_matchup(nha, lha, yA, yP, clT)

print(f"aug 統計 (action): {len(dMa_A)} players covered")
print(f"train-only (point/server): {len(dMa_T)} players covered")

In [ ]:
# Feature builders per task
def mkA_aug(Xb, nh, lh, idx):
    """Action features:用 aug 統計"""
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_AUG); Ft.index = idx
    return pd.concat([Xb[BASE], Ft,
                      player_feat(nh, dMa_A, gA_A, dMp_A, gP_A, idx),
                      matchup_feat(nh, lh, clA, cAA, cPA, dMa_A, gA_A, dMp_A, gP_A, idx)], axis=1)

def mkP_to(Xb, nh, lh, idx):
    """Point features:用 train-only 統計"""
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_TO); Ft.index = idx
    return pd.concat([Xb[BASE], Ft,
                      player_feat(nh, dMa_T, gA_T, dMp_T, gP_T, idx),
                      matchup_feat(nh, lh, clT, cAT, cPT, dMa_T, gA_T, dMp_T, gP_T, idx)], axis=1)

def mkAps_aug(Xb, nh, idx):
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_AUG); Ft.index = idx
    return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_A, gA_A, dMp_A, gP_A, idx)], axis=1)

def mkAps_to(Xb, nh, idx):
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_TO); Ft.index = idx
    return pd.concat([Xb[BASE], Ft, player_feat(nh, dMa_T, gA_T, dMp_T, gP_T, idx)], axis=1)

def mkS_to(Xb, idx):
    """Server features:沒 player/matchup(train-only transition only)"""
    Ft = apply_trans({k:Xb[k].to_numpy() for k in KEY}, T_TO); Ft.index = idx
    return pd.concat([Xb[BASE], Ft], axis=1)

## 9. ★ Priority 3:把 OLD 測試集當訓練資料加進去

這是 v14 的關鍵 — 過去把 old 只當「統計 prior」(player_dists、matchup),
現在直接把 old 樣本當 ground-truth training examples 丟進 LGBM/TabPFN/GRU。

📊 **為什麼這合法**:READ ME 明文寫「參賽者可自行決定是否使用該資訊進行研究或模型設計」
📊 **為什麼有效**:私人測試集裡 16% 的選手只在 old 出現過,給這些選手真實 labeled 訓練樣本能大幅幫助 model 預測他們


In [ ]:
# === 訓練集 features ===
# Action 用 aug,Point/Server 用 train-only
XaA  = mkA_aug(Xa,  nha, lha, Xa.index)
XaP  = mkP_to (Xa,  nha, lha, Xa.index)
XaS  = mkS_to (Xa,  Xa.index)
XsAa = mkAps_aug(Xs, nhs, Xs.index)
XsPt = mkAps_to (Xs, nhs, Xs.index)
XsS  = mkS_to  (Xs, Xs.index)

# === ★ OLD 樣本 features(同樣的 task gating)===
XaoA = mkA_aug(Xao, nhao, lhao, Xao.index)
XaoP = mkP_to (Xao, nhao, lhao, Xao.index)
XaoS = mkS_to (Xao, Xao.index)
Xao_smp_Aa = mkAps_aug(Xao_smp, nhao_s, Xao_smp.index)
Xao_smp_Pt = mkAps_to (Xao_smp, nhao_s, Xao_smp.index)
Xao_smp_S  = mkS_to  (Xao_smp, Xao_smp.index)

# === ★ Priority 3:concat train + old ===
XaA_c  = pd.concat([XaA,  XaoA],  ignore_index=True); yA_c = np.concatenate([yA, yAo])
XaP_c  = pd.concat([XaP,  XaoP],  ignore_index=True); yP_c = np.concatenate([yP, yPo])
XaS_c  = pd.concat([XaS,  XaoS],  ignore_index=True); yR_c = np.concatenate([yR, yRo])
XsAa_c = pd.concat([XsAa, Xao_smp_Aa], ignore_index=True); eA_c = np.concatenate([eA, eAo])
XsPt_c = pd.concat([XsPt, Xao_smp_Pt], ignore_index=True); eP_c = np.concatenate([eP, ePo])
XsS_c  = pd.concat([XsS,  Xao_smp_S],  ignore_index=True); eR_c = np.concatenate([eR, eRo])

print(f"LGBM train: {len(yA)} train + {len(yAo)} old = {len(yA_c)} total")
print(f"TabPFN sampled train: {len(eA)} train + {len(eAo)} old = {len(eA_c)} total")

## 10. 訓練 LGBM + TabPFN + GRU(全部用 train+old)

In [ ]:
print("Training LGBM...")
mA = lgbc().fit(XaA_c, yA_c)
mP = lgbc().fit(XaP_c, yP_c)
mR = lgbc(balanced=False).fit(XaS_c, yR_c)

print("Training TabPFN(這比較慢)...")
tA = ManyClassClassifier(estimator=TabPFNClassifier(device=DEV, ignore_pretraining_limits=True),
                          alphabet_size=TABPFN_MANY_CLASS_ALPHA, random_state=SEED).fit(XsAa_c, eA_c)
tP = TabPFNClassifier(device=DEV, ignore_pretraining_limits=True).fit(XsPt_c, eP_c)
tR = TabPFNClassifier(device=DEV, ignore_pretraining_limits=True).fit(XsS_c,  eR_c)

print("Training GRU(★ multi-seed × 2 architectures)...")
Ca_tr, Na_tr, La_tr, gyA_tr, gyP_tr, gyR_tr = build_seq(tr,  "all", tld, exclude_uids=EXCLUDE_UIDS)
Ca_o,  Na_o,  La_o,  gyA_o,  gyP_o,  gyR_o  = build_seq(old, "all", tld)
Ca = np.concatenate([Ca_tr, Ca_o]); Na = np.concatenate([Na_tr, Na_o]); La_ = np.concatenate([La_tr, La_o])
gyA = np.concatenate([gyA_tr, gyA_o]); gyP = np.concatenate([gyP_tr, gyP_o]); gyR = np.concatenate([gyR_tr, gyR_o])
print(f"  GRU train: {len(gyA_tr)} train + {len(gyA_o)} old = {len(gyA)} total")

Cte, Nte, Lte = build_seq(te, "sampled", tld, test_mode=True)
gA_v7, gP_v7, gR_v7 = multiseed_train_predict(GRUNetV7, Ca, Na, La_, gyA, gyP, gyR, Cte, Nte, Lte, SEEDS)
gA_v8, gP_v8, gR_v8 = multiseed_train_predict(GRUNetV8, Ca, Na, La_, gyA, gyP, gyR, Cte, Nte, Lte, SEEDS)
gA_ = GRU_BLEND*gA_v7 + (1-GRU_BLEND)*gA_v8
gP_ = GRU_BLEND*gP_v7 + (1-GRU_BLEND)*gP_v8
gR_ = GRU_BLEND*gR_v7 + (1-GRU_BLEND)*gR_v8
print("GRU done")

## 11. Ensemble + 預測 test

In [ ]:
def align(p, c, full):
    o = np.zeros((p.shape[0], len(full))); idx = {cc:i for i, cc in enumerate(c)}
    for j, cc in enumerate(full):
        if cc in idx: o[:, j] = p[:, idx[cc]]
    return o

# ensemble weights 從 cached v7 OOF 找出來的(per-task gating + multi-arch blend)
WA = (0.35, 0.25, 0.40); BA = 0.0    # action: LGBM 0.35 / TabPFN 0.25 / GRU 0.40, β=0
WP = (0.30, 0.40, 0.30); BP = 0.125  # point:  LGBM 0.30 / TabPFN 0.40 / GRU 0.30
WR = (0.15, 0.55, 0.30)               # server: LGBM 0.15 / TabPFN 0.55 / GRU 0.30

# Test features
XteA  = mkA_aug(Xte, nht, lht, Xte.index)
XteAa = mkAps_aug(Xte, nht, Xte.index)
XteP  = mkP_to(Xte, nht, lht, Xte.index)
XtePt = mkAps_to(Xte, nht, Xte.index)
XteS  = mkS_to(Xte, Xte.index)

PA = WA[0]*align(mA.predict_proba(XteA), mA.classes_, ACLS) + WA[1]*align(tA.predict_proba(XteAa), tA.classes_, ACLS) + WA[2]*gA_
PP = WP[0]*align(mP.predict_proba(XteP), mP.classes_, PCLS) + WP[1]*align(tP.predict_proba(XtePt), tP.classes_, PCLS) + WP[2]*gP_
PR = WR[0]*mR.predict_proba(XteS)[:, 1] + WR[1]*tR.predict_proba(XteS)[:, 1] + WR[2]*gR_
print(f"PA shape: {PA.shape}, PP shape: {PP.shape}, PR shape: {PR.shape}")

## 12. ★ Server Override(可選,但能拿到公開分 +0.07)

舊測試集裡有 1236 個 rally 的 `serverGetPoint` 真值(README 明文允許使用)。
post-hoc 替換 model 輸出,完全不影響 private(私人 24 場無 leak)。

選 `apply_override=True` → 公開 +0.07 加成(我們的 0.4341 來源)
選 `apply_override=False` → clean 版本(公開 ~0.36)


In [ ]:
apply_override = True   # 改成 False 可以拿到 clean 版

if apply_override:
    sgp_true = old.groupby('rally_uid').serverGetPoint.first().to_dict()
    PR_final = PR.copy(); n_ovr = 0
    for i, u in enumerate(uids):
        if int(u) in sgp_true:
            PR_final[i] = float(sgp_true[int(u)]); n_ovr += 1
    print(f"server override applied: {n_ovr}/{len(uids)} rallies")
else:
    PR_final = PR
    print("clean version (no override)")

## 13. 輸出 submission

In [ ]:
prA = np.array([(yA==c).mean() for c in ACLS])
prP = np.array([(yP==c).mean() for c in PCLS])

def decide(p, cls, pr, b, mask0):
    adj = p / np.clip(pr, 1e-9, None)**b
    if mask0:
        adj = adj.copy(); adj[:, 0] = -1e18
    return cls[adj.argmax(1)]

aid = decide(PA, ACLS, prA, BA, mask0=False).astype(int)
pid = decide(PP, PCLS, prP, BP, mask0=False).astype(int)
sub = pd.DataFrame({"rally_uid": uids, "actionId": aid, "pointId": pid,
                     "serverGetPoint": PR_final}).sort_values("rally_uid")
assert len(sub) == len(set(uids)) and not sub.rally_uid.duplicated().any()
assert sub.serverGetPoint.between(0, 1).all()
path = f"../submission_FINAL.csv"
sub.to_csv(path, index=False)
print(f"✅ wrote {path}")
print(f"   {len(sub)} rows, sgp_mean={sub.serverGetPoint.mean():.3f}")
print(f"\n上傳這個檔案到 AIdea 平台:{path}")